In [0]:
import random
random.seed(42)
 
genres = ["Pop", "Rock", "Hip-Hop", "Jazz", "Classical", "Electronic", "R&B", "Country"]
countries = ["US", "UK", "DE", "JP", "BR", "KR", "FR", "AU", "NG", "IN"]
 
artist_data = []
for i in range(50000):
    name = f"Artist_{i+1:05d}"
    genre = random.choice(genres)
    country = random.choice(countries)
    followers = random.randint(100, 5000000)
    monthly_listeners = random.randint(50, 2000000)
    verified = random.choice(["true", "false", "TRUE", "False", "yes", ""])
    join_date = f"202{random.randint(0,4)}-{random.randint(1,12):02d}-{random.randint(1,28):02d}"
 
    if random.random() < 0.02:
        followers = "NULL"
    if random.random() < 0.01:
        name = f'Artist "Special" {i}'
 
    artist_data.append((name, genre, country, str(followers), str(monthly_listeners), verified, join_date))
 
artist_columns = ["artist_name", "genre", "country", "followers", "monthly_listeners", "verified", "join_date"]
df_artists = spark.createDataFrame(artist_data, artist_columns)
df_artists.write.csv("/Volumes/workspace/default/demo/artists_csv", header=True, mode="overwrite", quote='"', escape='"')
print(f"✅ Artist catalog saved: {df_artists.count()} rows")

✅ Artist catalog saved: 50000 rows


In [0]:
event_data = []
for i in range(100000):
    event_data.append({
        "event_id": f"EVT-{i+1:07d}",
        "user_id": f"USR-{random.randint(1, 20000):06d}",
        "artist_id": f"ART-{random.randint(1, 50000):05d}",
        "track_id": f"TRK-{random.randint(1, 200000):06d}",
        "duration_seconds": random.randint(10, 400),
        "completed": random.choice([True, False]),
        "timestamp": f"2025-{random.randint(1,6):02d}-{random.randint(1,28):02d}T{random.randint(0,23):02d}:{random.randint(0,59):02d}:00Z",
        "device": random.choice(["mobile", "desktop", "smart_speaker", "tablet"]),
        "quality": random.choice(["low", "standard", "high", "lossless"]),
    })
 
df_events = spark.createDataFrame(event_data)
df_events.write.json("/Volumes/workspace/default/demo/events_json", mode="overwrite")
print(f"✅ Listening events saved: {df_events.count()} rows")

✅ Listening events saved: 100000 rows


In [0]:
user_data = []
for i in range(20000):
    user_data.append((
        f"USR-{i+1:06d}",
        f"user_{i+1}@email.com",
        random.choice(countries),
        random.choice(["free", "premium", "family", "student"]),
        random.randint(16, 75),
        f"202{random.randint(0,4)}-{random.randint(1,12):02d}-{random.randint(1,28):02d}",
    ))
 
user_columns = ["user_id", "email", "country", "subscription_tier", "age", "signup_date"]
df_users = spark.createDataFrame(user_data, user_columns)
df_users.write.parquet("/Volumes/workspace/default/demo/users_parquet", mode="overwrite")
print(f"✅ User profiles saved: {df_users.count()} rows")

✅ User profiles saved: 20000 rows


In [0]:
#Task 2a: Read without options (experience the problems)

df_bad = spark.read.csv("/Volumes/workspace/default/demo/artists_csv")
df_bad.show(5)
df_bad.printSchema()

+------------+----------+-------+---------+-----------------+--------+----------+
|         _c0|       _c1|    _c2|      _c3|              _c4|     _c5|       _c6|
+------------+----------+-------+---------+-----------------+--------+----------+
| artist_name|     genre|country|followers|monthly_listeners|verified| join_date|
|Artist_37501|   Country|     FR|  4224440|           878369|   false|2020-07-12|
|Artist_37502|      Jazz|     KR|  3195703|           886786|   False|2021-10-26|
|Artist_37503|Electronic|     AU|  4631877|          1622526|    TRUE|2024-02-17|
|Artist_37504|   Country|     FR|   402291|          1760215|    NULL|2020-09-09|
+------------+----------+-------+---------+-----------------+--------+----------+
only showing top 5 rows
root
 |-- _c0: string (nullable = true)
 |-- _c1: string (nullable = true)
 |-- _c2: string (nullable = true)
 |-- _c3: string (nullable = true)
 |-- _c4: string (nullable = true)
 |-- _c5: string (nullable = true)
 |-- _c6: string (nulla

Task 2a: The header (column names) is empty. The columns names have been integrated in the table as values in the first row. 
This has made that all datatypes are string.

In [0]:
#Task 2b: Read with proper options

df_artists = spark.read.csv(
    "/Volumes/workspace/default/demo/artists_csv",
    header=True,
    inferSchema=True,
    nullValue="NULL",
    quote='"',
    escape='"'
)
df_artists.show(5)
df_artists.printSchema()

+------------+----------+-------+---------+-----------------+--------+----------+
| artist_name|     genre|country|followers|monthly_listeners|verified| join_date|
+------------+----------+-------+---------+-----------------+--------+----------+
|Artist_37501|   Country|     FR|  4224440|           878369|   false|2020-07-12|
|Artist_37502|      Jazz|     KR|  3195703|           886786|   False|2021-10-26|
|Artist_37503|Electronic|     AU|  4631877|          1622526|    TRUE|2024-02-17|
|Artist_37504|   Country|     FR|   402291|          1760215|        |2020-09-09|
|Artist_37505|       R&B|     JP|    13687|          1781185|    TRUE|2024-06-27|
+------------+----------+-------+---------+-----------------+--------+----------+
only showing top 5 rows
root
 |-- artist_name: string (nullable = true)
 |-- genre: string (nullable = true)
 |-- country: string (nullable = true)
 |-- followers: integer (nullable = true)
 |-- monthly_listeners: integer (nullable = true)
 |-- verified: string 

In [0]:
#Task 2c: Read with an explicit schema
from pyspark.sql.types import *
from pyspark.sql.functions import *
import time

artist_schema = StructType([
    StructField("artist_name", StringType(), True),
    StructField("genre", StringType(), True),
    StructField("country", StringType(), True),
    StructField("followers", IntegerType(), True),
    StructField("monthly_listeners", IntegerType(), True),
    StructField("verified", StringType(), True),
    StructField("join_date", DateType(), True),
])

start = time.time()
df_infer = spark.read.csv("/Volumes/workspace/default/demo/artists_csv", header=True, inferSchema=True, nullValue="NULL")
time_infer = time.time() - start
 
start = time.time()
df_explicit = spark.read.csv("/Volumes/workspace/default/demo/artists_csv", header=True, schema=artist_schema,
                              nullValue="NULL", dateFormat="yyyy-MM-dd")
time_explicit = time.time() - start
 
print(f"inferSchema: {time_infer:.3f}s")
print(f"Explicit:    {time_explicit:.3f}s")

inferSchema: 0.000s
Explicit:    0.000s


In [0]:
#Task 2d: Handle the "verified" column inconsistency

df_artists_clean = df_explicit.withColumn(
    "verified",
    when(lower(col("verified")).isin("true", "yes"), True)
    .when(lower(col("verified")).isin("false", "no", ""), False)
    .otherwise(None)
    .cast("boolean")
)
df_artists_clean.groupBy("verified").count().show()

+--------+-----+
|verified|count|
+--------+-----+
|    true|24711|
|   false|25289|
+--------+-----+



In [0]:
# Task 3a: Read JSON and inspect the schema

start = time.time()
df_events = spark.read.json("/Volumes/workspace/default/demo/events_json")
time_json = time.time() - start
 
df_events.show(5, truncate=False)
df_events.printSchema()
print(f"JSON read: {time_json:.3f}s, {df_events.count()} rows")

+---------+---------+-------------+----------------+-----------+--------+--------------------+----------+----------+
|artist_id|completed|device       |duration_seconds|event_id   |quality |timestamp           |track_id  |user_id   |
+---------+---------+-------------+----------------+-----------+--------+--------------------+----------+----------+
|ART-24934|true     |smart_speaker|31              |EVT-0087501|lossless|2025-01-13T06:39:00Z|TRK-103155|USR-007496|
|ART-27577|false    |tablet       |159             |EVT-0087502|low     |2025-01-01T04:35:00Z|TRK-150476|USR-017584|
|ART-45551|false    |tablet       |112             |EVT-0087503|low     |2025-01-12T20:14:00Z|TRK-175082|USR-009722|
|ART-06237|true     |mobile       |87              |EVT-0087504|standard|2025-05-13T23:36:00Z|TRK-137546|USR-010413|
|ART-35491|false    |smart_speaker|191             |EVT-0087505|low     |2025-05-06T01:31:00Z|TRK-057302|USR-019034|
+---------+---------+-------------+----------------+-----------+

In [0]:
#Task 3b: Parse the timestamp

df_events_clean = df_events.withColumn(
    "event_timestamp",
    to_timestamp(col("timestamp"), "yyyy-MM-dd'T'HH:mm:ss'Z'")
).drop("timestamp")
 
df_events_clean.select("event_id", "event_timestamp").show(5, truncate=False)

+-----------+-------------------+
|event_id   |event_timestamp    |
+-----------+-------------------+
|EVT-0087501|2025-01-13 06:39:00|
|EVT-0087502|2025-01-01 04:35:00|
|EVT-0087503|2025-01-12 20:14:00|
|EVT-0087504|2025-05-13 23:36:00|
|EVT-0087505|2025-05-06 01:31:00|
+-----------+-------------------+
only showing top 5 rows


In [0]:
#Task 3c: Extract date components
df_events_enriched = df_events_clean \
    .withColumn("event_date", to_date(col("event_timestamp"))) \
    .withColumn("event_hour", hour(col("event_timestamp"))) \
    .withColumn("event_month", month(col("event_timestamp")))
 
df_events_enriched.select("event_id", "event_date", "event_hour", "event_month").show(5)

+-----------+----------+----------+-----------+
|   event_id|event_date|event_hour|event_month|
+-----------+----------+----------+-----------+
|EVT-0087501|2025-01-13|         6|          1|
|EVT-0087502|2025-01-01|         4|          1|
|EVT-0087503|2025-01-12|        20|          1|
|EVT-0087504|2025-05-13|        23|          5|
|EVT-0087505|2025-05-06|         1|          5|
+-----------+----------+----------+-----------+
only showing top 5 rows


In [0]:
#Task 4a: Read Parquet and appreciate the ease

start = time.time()
df_users = spark.read.parquet("/Volumes/workspace/default/demo/users_parquet")
time_parquet = time.time() - start
 
df_users.show(5)
df_users.printSchema()
print(f"Parquet read: {time_parquet:.3f}s, {df_users.count()} rows")

+----------+-------------------+-------+-----------------+---+-----------+
|   user_id|              email|country|subscription_tier|age|signup_date|
+----------+-------------------+-------+-----------------+---+-----------+
|USR-007501|user_7501@email.com|     UK|             free| 23| 2022-10-23|
|USR-007502|user_7502@email.com|     BR|          student| 43| 2022-02-23|
|USR-007503|user_7503@email.com|     AU|          premium| 31| 2024-05-06|
|USR-007504|user_7504@email.com|     BR|           family| 25| 2023-08-05|
|USR-007505|user_7505@email.com|     AU|          premium| 25| 2022-02-10|
+----------+-------------------+-------+-----------------+---+-----------+
only showing top 5 rows
root
 |-- user_id: string (nullable = true)
 |-- email: string (nullable = true)
 |-- country: string (nullable = true)
 |-- subscription_tier: string (nullable = true)
 |-- age: long (nullable = true)
 |-- signup_date: string (nullable = true)

Parquet read: 0.000s, 20000 rows


In [0]:
#Task 4b: Demonstrate column pruning

start = time.time()
df_all = spark.read.parquet("/Volumes/workspace/default/demo/users_parquet")
_ = df_all.count()
time_all = time.time() - start
 
start = time.time()
df_two = spark.read.parquet("/Volumes/workspace/default/demo/users_parquet").select("user_id", "subscription_tier")
_ = df_two.count()
time_two = time.time() - start
 
print(f"All columns:  {time_all:.3f}s")
print(f"Two columns:  {time_two:.3f}s")

All columns:  1.207s
Two columns:  0.920s


Part 5: Format Comparison Summary.
Csv format is the one that requires more data cleaning. The easiest format to work with is parquet.

In [0]:
#Part 6: Unified Output 
df_enriched_listens = df_events_enriched \
    .join(df_users.select("user_id", "country", "subscription_tier", "age"),
          on="user_id", how="left") \
    .join(df_artists_clean.select(
              col("artist_name"),
              col("genre"),
              col("country").alias("artist_country"),
              col("followers")
          ).withColumn("artist_id",
              concat(lit("ART-"), lpad(monotonically_increasing_id().cast("string"), 5, "0"))
          ),
          on="artist_id", how="left")
    

df_enriched_listens.coalesce(4) \
    .write.parquet(
        "/Volumes/workspace/default/demo/unified_listening_analytics",
        mode="overwrite",
        partitionBy=["event_month"],
        compression="snappy"
    )
 
result = spark.read.parquet("/Volumes/workspace/default/demo/unified_listening_analytics")
print(f"✅ Unified dataset: {result.count()} rows, {len(result.columns)} columns")
result.show(5)

✅ Unified dataset: 143743 rows, 19 columns
+---------+----------+---------+-------+----------------+-----------+--------+----------+-------------------+----------+----------+-------+-----------------+---+--------------------+-------+--------------+---------+-----------+
|artist_id|   user_id|completed| device|duration_seconds|   event_id| quality|  track_id|    event_timestamp|event_date|event_hour|country|subscription_tier|age|         artist_name|  genre|artist_country|followers|event_month|
+---------+----------+---------+-------+----------------+-----------+--------+----------+-------------------+----------+----------+-------+-----------------+---+--------------------+-------+--------------+---------+-----------+
|ART-42949|USR-016784|     true|desktop|             335|EVT-0065887|lossless|TRK-057530|2025-02-06 00:21:00|2025-02-06|         0|     NG|             free| 67|        Artist_31251|   Jazz|            IN|  2694981|          2|
|ART-42949|USR-016784|     true|desktop|     